# zvec Benchmark on Colab

In [ ]:
# Clone zvec
!git clone -b sprint-gpu-optimization https://github.com/cluster2600/zvec.git
%cd zvec

In [ ]:
# Install faiss-gpu
!pip install faiss-gpu-cu12 -q

In [ ]:
# Check GPU
import faiss
print(f"FAISS GPUs: {faiss.get_num_gpus()}")

In [ ]:
# Add python path
import sys
sys.path.insert(0, '/content/zvec/python')

# Import zvec
import zvec
print("✓ zvec imported")
print("✓ Available:", dir(zvec))

In [ ]:
# Test backends
from zvec.backends import quantization
print("✓ quantization loaded")

In [ ]:
# Test PQ
import numpy as np
from zvec.backends.quantization import PQEncoder

np.random.seed(42)
vectors = np.random.random((1000, 128)).astype(np.float32)

encoder = PQEncoder(m=8, nbits=8, k=256)
encoder.train(vectors)
codes = encoder.encode(vectors)

print(f"✓ PQ Test: {vectors.shape} -> codes {codes.shape}")
print(f"Compression: {vectors.nbytes / codes.nbytes:.1f}x")

In [ ]:
# Test FAISS GPU
import faiss
import numpy as np

dim = 128
vectors = np.random.random((10000, dim)).astype(np.float32)

# Create CPU index
index = faiss.IndexFlatL2(dim)
index.add(vectors)

# Move to GPU
gpu_resources = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(gpu_resources, 0, index)

query = np.random.random((5, dim)).astype(np.float32)
distances, indices = gpu_index.search(query, k=10)

print(f"✓ FAISS GPU: {distances.shape}")